> ⚠️ **作業中 (Work in Progress)**: このノートブックは現在開発中です。一部のコードが不完全であったり変更される可能性があります。

## 📋 目次

- [Control Plane概要](#control-plane-개요)
- [Fleet Overview](#fleet-overview)
- [Assets 관리](#assets-관리)
- [Compliance 및 セキュリティ](#compliance-및-セキュリティ)
- [Quota 관리](#quota-관리)
- [Admin 機能](#admin-機能)

## 🎯 学習目標

- Control Plane의 역할과 重要성 이해
- Fleet Overview를 통한 전체 시스템 モニタリング
- Assets(エージェント, モデル, 도구) 관리 방법
- 컴플라이언스 및 セキュリティ 設定 構成
- Quota 및 Rate Limiting 관리
- プロジェクト 및 使用자 権限 관리

## ⏱️ 予想所要時間

約10分

## Control Plane概要

### Control Plane이란?

Control Plane은 Microsoft Foundry의 중앙 관리 센터로, 모든 AI リソース를 통합 관리하고 モニタリング합니다.

```
Control Plane = モニタリング + 관리 + セキュリティ + 거버넌스
```

### 주요 機能 영역

```
┌─────────────────────────────────────────┐
│         Control Plane                   │
├─────────────────────────────────────────┤
│ Fleet Overview   │ 전체 시스템 대시보드   │
│ Assets          │ リソース管理           │
│ Compliance      │ セキュリティ 및 정책          │
│ Quota           │ 할당량 및 제한        │
│ Admin           │ プロジェクト 및 権限      │
└─────────────────────────────────────────┘
```

### 왜 重要한가?

프로덕션 環境에서 次へ을 보장합니다:
- 📊 **가시성**: 모든 リソース의 状態와 パフォーマンス 파악
- 🛡️ **セキュリティ**: 취약점과 위협 조기 발견
- 💰 **비용 관리**: リソース 使用량과 비용 최적화
- ⚖️ **컴플라이언스**: 규정 준수 状態 モニタリング
- 🚨 **通知**: 문제 발생 시 즉각 대응

### Portal vs SDK

| 機能 | Portal | SDK (Python) |
|-----|--------|-------------|
| Fleet Overview | ✅ 대시보드 제공 | ⚠️ 제한적 取得 |
| Assets 取得 | ✅ 시각화 | ✅ 프ログ래밍 방식 |
| Compliance モニタリング | ✅ 실時間 대시보드 | ❌ 지원 안 함 |
| Quota 관리 | ✅ Portal에서 관리 | ❌ 取得만 가능 |
| Admin 設定 | ✅ RBAC 관리 | ❌ 지원 안 함 |

**💡 推奨 사항:**
- **Portal**: 실時間 モニタリング, セキュリティ/컴플라이언스 確認, Quota 관리
- **SDK**: 자동화, CI/CD 파이프라인, 프ログ래밍 방식 取得

## 環境設定

In [ ]:
# 環境 変数 ロード
import json
import os
import subprocess

# PATH 環境変数 設定 (Azure CLI를 찾을 수 있도록)
possible_paths = [
    "/opt/homebrew/bin",  # macOS (Apple Silicon)
    "/usr/local/bin",     # macOS (Intel) / Linux
    "/usr/bin",           # Linux / GitHub Codespaces
    "/home/linuxbrew/.linuxbrew/bin"  # Linux Homebrew
]

az_path = None
try:
    result = subprocess.run(['which', 'az'], capture_output=True, text=True)
    if result.returncode == 0:
        az_path = os.path.dirname(result.stdout.strip())
except:
    pass

paths_to_add = []
if az_path and az_path not in os.environ.get("PATH", ""):
    paths_to_add.append(az_path)
else:
    for path in possible_paths:
        if os.path.exists(path) and path not in os.environ.get("PATH", ""):
            paths_to_add.append(path)

if paths_to_add:
    new_path = ":".join(paths_to_add) + ":" + os.environ.get("PATH", "")
    os.environ["PATH"] = new_path

# 前へ ノート북에서 保存한 設定 ファイル ロード
config_file = ".foundry_config.json"
try:
    with open(config_file, 'r') as f:
        config = json.load(f)
    
    # 環境 変数 設定
    FOUNDRY_NAME = config.get("FOUNDRY_NAME")
    RESOURCE_GROUP = config.get("RESOURCE_GROUP")
    LOCATION = config.get("LOCATION")
    TENANT_ID = config.get("TENANT_ID")
    PROJECT_NAME = config.get("PROJECT_NAME", "proj-default")
    PROJECT_ENDPOINT = config.get("FOUNDRY_ENDPOINT")
    
    # 環境 変数로도 設定 (다른 도구들이 使用할 수 있도록)
    os.environ["FOUNDRY_NAME"] = FOUNDRY_NAME
    os.environ["LOCATION"] = LOCATION
    os.environ["RESOURCE_GROUP"] = RESOURCE_GROUP
    os.environ["AZURE_SUBSCRIPTION_ID"] = config.get("AZURE_SUBSCRIPTION_ID", "")
    os.environ["_ENDPOINT"] = PROJECT_ENDPOINT
    os.environ["PROJECT_ENDPOINT"] = PROJECT_ENDPOINT
    
    print(f"✅ 設定 ファイル '{config_file}'에서 環境 変数를 ロード했습니다.")
    print(f"\n📌 Foundry Name: {FOUNDRY_NAME}")
    print(f"📌 Resource Group: {RESOURCE_GROUP}")
    print(f"📌 Location: {LOCATION}")
    print(f"📌 プロジェクト エンドポイント: {PROJECT_ENDPOINT}")
    
except FileNotFoundError:
    print(f"⚠️ '{config_file}' ファイル을 찾을 수 없습니다.")
    print("💡 01-setup.ipynb를 먼저 実行하여 環境을 設定하세요.")
    raise

# 必須パッケージのインストール
%pip install -q azure-ai-projects azure-identity

from azure.ai.projects import AIProjectClient
from azure.identity import DefaultAzureCredential

print(f"\n💡 使用할 プロジェクト エンドポイント: {PROJECT_ENDPOINT}")


## リソース モニタリング

デプロイ된 モデル과 エージェント를 取得합니다.

In [ ]:
# Fleet Overview - デプロイ된 エージェント 状態 確認
credential = DefaultAzureCredential()
client = AIProjectClient(endpoint=PROJECT_ENDPOINT, credential=credential)

try:
    # デプロイ된 エージェント リスト 取得
    agents = list(client.agents.list())
    
    print("🚀 Fleet Overview - Running Agents")
    print("=" * 80)
    
    if not agents:
        print("\n⚠️ デプロイ된 エージェント가 없습니다.")
        print("💡 03-agents.ipynb를 먼저 実行하여 エージェント를 作成하세요.")
    else:
        print(f"\n📊 合計 {len(agents)}개의 エージェント가 実行 중입니다:\n")
        
        for i, agent in enumerate(agents, 1):
            print(f"{i}. {agent.name}")
            print(f"   ID: {agent.id}")
            
            # Model 情報 (속성명이 다를 수 있음)
            model_name = getattr(agent, 'model', None) or getattr(agent, 'model_id', 'N/A')
            print(f"   Model: {model_name}")
            
            if hasattr(agent, 'tools') and agent.tools:
                tools_str = ", ".join([t.get('type', 'unknown') for t in agent.tools])
                print(f"   Tools: {tools_str}")
            
            if hasattr(agent, 'created_at'):
                print(f"   Created: {agent.created_at}")
            
            print(f"   Status: ✅ Active")
            print("-" * 80)
        
        print(f"\n✅ Fleet Overview 取得 完了!")
        
except Exception as e:
    print(f"⚠️ Fleet 情報 取得 失敗: {e}")
    print("\n💡 解決方法:")
    print("   1. Azure에 ログイン했는지 確認 (az login)")
    print("   2. FOUNDRY_NAME이 올바른지 確認")
    print("   3. プロジェクト에 エージェント가 作成되었는지 確認")

## Quota 確認

モデル별 할당량과 使用률을 確認합니다.

**⚠️ 参考**: 자세한 Quota 관리는 Azure Portal에서만 가능합니다.
- Operate > Quota에서 실時間 使用률 確認
- Quota 증가 リクエスト

In [ ]:
# Assets 관리 - モデル, 接続, 도구 確認

print("📦 Assets 관리")
print("=" * 80)

try:
    # 1. 接続(Connections) 取得
    print("\n1️⃣ Connections (モデル 接続):")
    print("-" * 80)
    
    connections = list(client.connections.list())
    
    if not connections:
        print("⚠️ 接続된 モデル이 없습니다.")
        print("💡 Portal에서 モデル을 デプロイ하세요: https://ai.azure.com")
    else:
        for conn in connections:
            print(f"\n✅ {conn.name}")
            if hasattr(conn, 'connection_type'):
                print(f"   Type: {conn.connection_type}")
            if hasattr(conn, 'endpoint_url'):
                print(f"   Endpoint: {conn.endpoint_url}")
    
    # 2. エージェント リスト (Assets)
    print("\n\n2️⃣ Agents:")
    print("-" * 80)
    
    agents = list(client.agents.list())
    
    if agents:
        for agent in agents:
            print(f"\n✅ {agent.name}")
            print(f"   ID: {agent.id}")
            
            # Model 情報 (속성명이 다를 수 있음)
            model_name = getattr(agent, 'model', None) or getattr(agent, 'model_id', 'N/A')
            print(f"   Model: {model_name}")
    else:
        print("⚠️ 作成된 エージェント가 없습니다.")
    
    print("\n" + "=" * 80)
    print("✅ Assets 取得 完了!")
    
    # 3. リソース 통계 시각화
    print("\n\n3️⃣ リソース 통계 시각화:")
    print("-" * 80)
    
    # matplotlib 설치
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "matplotlib"])
    
    import matplotlib.pyplot as plt
    
    # 接続 및 エージェント 수 시각화
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # 接続 タイプ별 분포
    if connections:
        connection_types = {}
        for conn in connections:
            if hasattr(conn, 'connection_type'):
                conn_type = str(conn.connection_type)
                connection_types[conn_type] = connection_types.get(conn_type, 0) + 1
        
        if connection_types:
            ax1.bar(connection_types.keys(), connection_types.values(), color='skyblue')
            ax1.set_title('Connection Types Distribution', fontsize=14, fontweight='bold')
            ax1.set_xlabel('Connection Type')
            ax1.set_ylabel('Count')
            ax1.tick_params(axis='x', rotation=45)
    else:
        ax1.text(0.5, 0.5, 'No connections available', ha='center', va='center', fontsize=12)
        ax1.set_title('Connection Types Distribution', fontsize=14, fontweight='bold')
    
    # エージェント モデル 분포
    if agents:
        agent_models = {}
        for agent in agents:
            model_name = getattr(agent, 'model', None) or getattr(agent, 'model_id', 'Unknown')
            agent_models[model_name] = agent_models.get(model_name, 0) + 1
        
        ax2.bar(agent_models.keys(), agent_models.values(), color='lightcoral')
        ax2.set_title('Agent Models Distribution', fontsize=14, fontweight='bold')
        ax2.set_xlabel('Model')
        ax2.set_ylabel('Count')
        ax2.tick_params(axis='x', rotation=45)
    else:
        ax2.text(0.5, 0.5, 'No agents available', ha='center', va='center', fontsize=12)
        ax2.set_title('Agent Models Distribution', fontsize=14, fontweight='bold')
    
    plt.tight_layout()
    plt.show()
    
    print("\\n📊 リソース 통계 시각화 完了!")
    
except Exception as e:
    print(f"\n⚠️ Assets 取得 失敗: {e}")
    print("💡 Portal에서 確認: https://ai.azure.com > Build > Assets")

## リソース 정리 (選択사항)

워크샵 終了 후 비용 절감을 위해 リソース를 削除할 수 있습니다.

In [ ]:
# Quota 관리 情報
print("📊 Quota 및 Rate Limiting 情報")
print("=" * 80)
print("\n💡 Quota 및 Rate Limiting은 주로 Azure Portal에서 관리됩니다.")
print("\n確認 방법:")
print("   1. Azure Portal: https://portal.azure.com")
print("   2. AI Services リソース 選択")
print("   3. 'Quota' 또는 'Usage + quotas' 메뉴 確認")
print("\n주요 確認 항목:")
print("   - Token Per Minute (TPM): 분당 처리 가능한 トークン 수")
print("   - Requests Per Minute (RPM): 분당 처리 가능한 リクエスト 수")
print("   - Provisioning Throughput Units (PTU): 프로비저닝된 처리 단위")
print("\n" + "=" * 80)
print("✅ Portal에서 실時間 quota 使用량을 確認하세요!")

## 📚 追加リソース

- [Microsoft Foundry Control Plane](https://learn.microsoft.com/en-us/azure/ai-foundry/control-plane/overview?view=foundry)
- [Agent 헬스 및 パフォーマンス モニタリング](https://learn.microsoft.com/en-us/azure/ai-foundry/control-plane/monitoring-across-fleet?view=foundry)
- [Custom Agent 등록](https://learn.microsoft.com/en-us/azure/ai-foundry/control-plane/register-custom-agent?view=foundry)
- [Guardrail Policy 作成](https://learn.microsoft.com/en-us/azure/ai-foundry/control-plane/quickstart-create-guardrail-policy?view=foundry)
- [규정 준수 및 セキュリティ 관리](https://learn.microsoft.com/en-us/azure/ai-foundry/control-plane/how-to-manage-compliance-security?view=foundry)
- [モデル 비용 및 パフォーマンス 최적화](https://learn.microsoft.com/en-us/azure/ai-foundry/control-plane/how-to-optimize-cost-performance?view=foundry)

## 마무리

### 축하합니다! 🎉

Microsoft Foundry Hands-on Workshop의 모든 モジュール을 完了했습니다!

### 학습한 내용 요약

```
✅ Module 01: 環境設定
   - Resource Group 및 Foundry リソース 作成

✅ Module 02: モデル 및 デプロイ
   - モデル 탐색, デプロイ, Model Router 構成

✅ Module 03: エージェント 개발
   - 다양한 タイプ의 エージェント 作成 및 デプロイ

✅ Module 04: Foundry IQ
   - AI Search 및 Blob Storage 기반 Knowledge Base 구축

✅ Module 05: ワークフロー
   - Sequential, Group Chat, Human-in-loop ワークフロー 구현

✅ Module 06: 評価
   - エージェント 품질 評価 및 개선

✅ Module 07: Control Plane
   - 프로덕션 モニタリング 및 관리
```

### 次のステップ

이제 次へ을 수행할 준비가 되었습니다:

1. **프로덕션 デプロイ**
   - 실제 애플리케이션에 Foundry 통합
   - CI/CD 파이프라인 구축
   - モニタリング 및 通知 設定

2. **고급 機能 탐색**
   - Custom tools 및 MCP servers 개발
   - Multi-agent 협업 패턴
   - Fine-tuning 및 モデル 커스터마이징

3. **커뮤니티 참여**
   - [Microsoft Tech Community](https://techcommunity.microsoft.com) 포럼 참여
   - [GitHub Samples](https://github.com/Azure-Samples) 기여
   - 使用 사례 공유